# 02 Methodology And Objective Weeks

This is the light setup checkpoint directly before the benchmark notebooks.

What is frozen here:
- daily rolling-origin evaluation with origin at `08:00` on `D-1`
- forecast horizon `D` through `D+4`
- UTC as the internal storage timezone
- the active feature-stage ladder from `FS0` through `FS4`
- the rule that thesis case weeks must come from **test actual prices only**

This notebook stays lightweight. It does not launch model benchmarks, but it can refresh the objective case-week selection artifact used later in the final reporting notebook.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


## Optional refresh hook


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:

    estimate = estimate_run_duration_seconds(output_root, "case_week_selection")
    if estimate is not None:
        print(
            "Heavy rerun warning: latest comparable run "
            f"{estimate['run_id']} suggests about {format_duration(float(estimate['estimate_seconds']))}."
        )
    else:
        print("Heavy rerun warning: no comparable runtime estimate was found for this stage.")

    command = [sys.executable, str(PACKAGE_ROOT / "run_case_week_selection.py")]

    started = time.perf_counter()
    run_command_with_live_output(command)
    elapsed_seconds = time.perf_counter() - started
    print(f"Actual wall-clock time: {format_duration(elapsed_seconds)}")
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True only when you are ready to execute the finalized pipeline.")


## Active methodology snapshot

The tables below are the single source of truth for the current DAM ladder, model status, shortlisting policy, and tuning cadence.


In [ ]:
display(Markdown(f"**FS1 foundation note.** {STARTER_ENDOGENOUS_FEATURE_NOTE}"))
display(feature_stage_policy_frame())
display(model_status_frame())
display(shortlisting_policy_frame())
display(tuning_cadence_frame())


## Objective test-week selection

The required thesis cases are:
- one typical winter week
- one typical summer week
- one high-volatility week

The selection remains objective and is never hand-picked from model outputs.


In [ ]:
try:
    selected_weeks_run, selected_weeks = load_selected_case_weeks(output_root)
    print(selected_weeks_run)
    required_categories = ["typical_winter", "typical_summer", "high_volatility"]
    if "category" in selected_weeks.columns:
        selected_weeks_view = selected_weeks[selected_weeks["category"].isin(required_categories)].copy()
        if selected_weeks_view.empty:
            selected_weeks_view = selected_weeks.copy()
    else:
        selected_weeks_view = selected_weeks.copy()
    display(selected_weeks_view)
except FileNotFoundError:
    print("No saved objective week selection artifact exists yet.")
